# Excel Re-ID — 3-step workflow

**Step 1 →** Run the first two cells. Download `reid_template.xlsx`.

**Step 2 →** Open the Excel. For each row, jump to `check_f1` in the annotated video and type the player's shirt number in the `canonical_id` column. Rows for the same person at different time periods → same `canonical_id`. Drop false detections → `canonical_id = 0`.

**Step 3 →** Run the last two cells. Upload your filled Excel. Get `per_frame_tracks_clean.csv`.

---
**No frame hunting needed** — the template already has the frame boundaries and check frames. You only fill in `canonical_id`.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
TRACKS_CSV   = "/content/per_frame_tracks_HILHAZ_half1.csv"   # input tracker CSV
CLEAN_CSV    = "/content/per_frame_tracks_clean.csv"          # output clean CSV
TEMPLATE_XL  = "/content/reid_template.xlsx"                  # template to fill in

# Swap detection thresholds (pixel space — no homography needed)
SWAP_BODY_MULT = 2.5   # flag swap when jump > this × player body height
SWAP_MIN_PX    = 180   # …and at least this many pixels
SPLIT_ON_CLS   = True  # also split on class_id flip (player ↔ referee)

# Gap-fill: interpolate short gaps per canonical ID
MAX_GAP = 25

In [ ]:
# ── STEP 1 & 2: Detect segments → export template Excel ───────────────────────
import subprocess, sys
for pkg in ["pandas", "openpyxl", "numpy"]:
    try: __import__(pkg)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import pandas as pd
import numpy as np
from google.colab import files as colab_files

BALL_CLASS   = 0
GK_CLASS     = 1
PLAYER_CLASS = 2
REF_CLASS    = 3
PEOPLE       = (GK_CLASS, PLAYER_CLASS, REF_CLASS)
_CLS_NAME    = {0:"ball", 1:"GK", 2:"player", 3:"referee"}

# ── load CSV ──────────────────────────────────────────────────────────────────
df = pd.read_csv(TRACKS_CSV, encoding_errors="replace", low_memory=False)
print(f"Loaded {len(df):,} rows · tracks: {sorted(df.display_track_id.unique())}")

# De-duplicate (frame, tracker_id) — keeps highest-confidence box
_before = len(df)
if "conf" in df.columns:
    df = df.sort_values("conf", ascending=False)
df = df.drop_duplicates(["frame", "display_track_id"], keep="first")
df = df.sort_values(["frame", "display_track_id"]).reset_index(drop=True)
_dups = _before - len(df)
if _dups:
    print(f"Removed {_dups} duplicate rows — the over-split fix.")

df["cx"] = (df["x1"] + df["x2"]) / 2
df["cy"] = (df["y1"] + df["y2"]) / 2
FPS = 30.0  # overridden below if video is opened

# ── detect swaps (pixel space) ────────────────────────────────────────────────
people = df[df["class_id"].isin(PEOPLE)].copy()
print(f"People rows: {len(people):,}  |  tracker IDs: {people.display_track_id.nunique()}")

def _detect_swaps(ppl):
    swaps = {}
    for tid, g in ppl.groupby("display_track_id"):
        g = g.sort_values("frame")
        frames = g["frame"].values
        cls    = g["class_id"].values
        cx     = g["cx"].values
        cy     = g["cy"].values
        h      = (g["y2"].values - g["y1"].values).astype(float)
        split_at = []
        for i in range(1, len(frames)):
            gap = int(frames[i] - frames[i-1])
            if gap > 10:
                continue
            jump = float(np.hypot(cx[i] - cx[i-1], cy[i] - cy[i-1]))
            body = max(1.0, 0.5 * (h[i] + h[i-1]))
            if jump > max(SWAP_MIN_PX, SWAP_BODY_MULT * body):
                split_at.append(int(frames[i]))
                continue
            if SPLIT_ON_CLS and cls[i] != cls[i-1]:
                split_at.append(int(frames[i]))
        if split_at:
            swaps[int(tid)] = sorted(set(split_at))
    return swaps

swaps = _detect_swaps(people)
print(f"Swap points: {sum(len(v) for v in swaps.values())} across {len(swaps)} tracker IDs")

# ── build segments ────────────────────────────────────────────────────────────
def _mode(s):
    m = s.mode(); return int(m.iloc[0]) if len(m) else -1

rows = []
for tid, g in people.groupby("display_track_id"):
    tid = int(tid)
    g   = g.sort_values("frame")
    all_frames = g["frame"].values
    boundaries = ([all_frames[0]] + sorted(swaps.get(tid, []))
                  + [all_frames[-1] + 1])
    for seg_i in range(len(boundaries) - 1):
        f0, f1 = boundaries[seg_i], boundaries[seg_i+1] - 1
        sg = g[(g["frame"] >= f0) & (g["frame"] <= f1)]
        if len(sg) == 0:
            continue
        mc = _mode(sg["class_id"])
        mt = _mode(sg["team_id"])
        # three check frames spread across the segment
        cf = sorted(set([int(f0),
                         int((f0 + f1) // 2),
                         int(f1)]))
        rows.append({
            "tracker_id":  tid,
            "frame_start": int(f0),
            "frame_end":   int(f1),
            "n_detections": len(sg),
            "duration_s":  round((f1 - f0) / FPS, 1),
            "auto_role":   _CLS_NAME.get(mc, "?"),
            "auto_team":   ("home" if mt == 0 else ("away" if mt == 1 else "?")),
            "check_f1":    cf[0],
            "check_f2":    cf[1] if len(cf) > 1 else cf[0],
            "check_f3":    cf[2] if len(cf) > 2 else cf[-1],
            # ── user fills these two ──
            "canonical_id": "",
            "team":         mt if mt in (0, 1) else "",
        })

seg_df = pd.DataFrame(rows)
print(f"\nSegments to label: {len(seg_df)}  "
      f"(was {people.display_track_id.nunique()} raw IDs before split)")

# ── export template ───────────────────────────────────────────────────────────
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "reid"

HDR_FILL  = PatternFill("solid", fgColor="1E3A5F")
FILL_COL  = PatternFill("solid", fgColor="FFF2CC")  # yellow = user must fill
HINT_FILL = PatternFill("solid", fgColor="E8F4E8")  # light green = auto-filled
HDR_FONT  = Font(bold=True, color="FFFFFF", size=10)
THIN      = Side(style="thin", color="CCCCCC")

cols = list(seg_df.columns)
# Column widths
widths = {"tracker_id":12,"frame_start":13,"frame_end":12,"n_detections":14,
          "duration_s":12,"auto_role":10,"auto_team":10,
          "check_f1":10,"check_f2":10,"check_f3":10,
          "canonical_id":15,"team":10}

# Header row
for ci, col in enumerate(cols, 1):
    cell = ws.cell(1, ci, col.upper().replace("_"," "))
    cell.fill  = HDR_FILL
    cell.font  = HDR_FONT
    cell.alignment = Alignment(horizontal="center")
    ws.column_dimensions[get_column_letter(ci)].width = widths.get(col, 12)

# Data rows
user_cols = {"canonical_id", "team"}
for ri, row in seg_df.iterrows():
    for ci, col in enumerate(cols, 1):
        val  = row[col]
        cell = ws.cell(ri + 2, ci, "" if pd.isna(val) else val)
        cell.alignment = Alignment(horizontal="center")
        cell.border = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
        if col in user_cols:
            cell.fill = FILL_COL
            cell.font = Font(bold=True, size=11)
        else:
            cell.fill = HINT_FILL

# Freeze pane + instruction row
ws.freeze_panes = "A2"
ws.row_dimensions[1].height = 22

# Instructions sheet
wi = wb.create_sheet("INSTRUCTIONS")
wi["A1"] = "HOW TO FILL IN 'reid' sheet"
wi["A1"].font = Font(bold=True, size=14)
instrs = [
    "",
    "YELLOW columns are yours to fill. Green columns are auto-filled — do not change.",
    "",
    "CANONICAL_ID  →  The player's SHIRT NUMBER (integer).",
    "               Same person in different segments = same shirt number.",
    "               False detections / trash boxes = 0 (will be dropped).",
    "",
    "TEAM          →  0 = home team,  1 = away team.  Leave blank if unsure.",
    "",
    "check_f1/f2/f3  →  Jump to these frame numbers in the annotated video",
    "               to identify who the tracker box is following at that time.",
    "",
    "EXAMPLES:",
    "  Tracker 6, frames 4270-4387:  check_f1=4270 → it's shirt #17 → canonical_id=17",
    "  Tracker 6, frames 4388-5000:  check_f1=4388 → it's shirt #21 → canonical_id=21",
    "  Tracker 2, frames 0-100:      check_f1=0    → false/blurry   → canonical_id=0",
]
for i, line in enumerate(instrs, 2):
    wi.cell(i, 1, line)
wi.column_dimensions["A"].width = 80

wb.save(TEMPLATE_XL)
print(f"\n✅ Template saved → {TEMPLATE_XL}")
print(f"   {len(seg_df)} rows to label (yellow columns: canonical_id + team)")
colab_files.download(TEMPLATE_XL)
print("\nNow: open the Excel, fill in canonical_id for each row, save, come back here.")

In [ ]:
# ── STEP 3: Upload filled Excel → apply → save clean CSV ──────────────────────
print("Select your filled reid_template.xlsx …")
uploaded = colab_files.upload()
xl_path  = "/content/" + list(uploaded.keys())[0]
print("Uploaded:", xl_path)

In [ ]:
# ── Apply mapping ──────────────────────────────────────────────────────────────
mapping = pd.read_excel(xl_path, sheet_name="reid")
# Normalise column names (case-insensitive)
mapping.columns = [c.strip().lower().replace(" ", "_") for c in mapping.columns]

assert "canonical_id" in mapping.columns, (
    f"Column 'canonical_id' not found. Got: {list(mapping.columns)}")

# Show summary of what was filled in
print("Filled segments:", mapping["canonical_id"].notna().sum(), "/", len(mapping))
print("Blank canonical_id rows (will keep raw tracker ID):",
      mapping["canonical_id"].isna().sum())

# Drop rows with no canonical_id entry (treat as 'keep raw')
mapping["canonical_id"] = pd.to_numeric(mapping["canonical_id"], errors="coerce")

# ── Apply row by row ───────────────────────────────────────────────────────────
_ROLE_TEAM = {"home": 0, "away": 1}

out = df.copy()
ppl_mask = out["class_id"].isin(PEOPLE)
out["new_id"]   = out["display_track_id"]
out["new_team"] = out["team_id"]
drop_idx = []

for _, m in mapping.iterrows():
    if pd.isna(m["canonical_id"]):
        continue
    canon = int(m["canonical_id"])
    tid   = int(m["tracker_id"])
    f0    = int(m["frame_start"])
    f1    = int(m["frame_end"])

    sel = (ppl_mask &
           (out["display_track_id"] == tid) &
           (out["frame"] >= f0) &
           (out["frame"] <= f1))

    if canon == 0:               # 0 = drop this segment
        drop_idx.extend(out.index[sel].tolist())
        continue

    out.loc[sel, "new_id"] = canon

    # override team if the user filled it in
    if "team" in m.index and not pd.isna(m["team"]):
        try:
            t = int(float(m["team"]))
            out.loc[sel, "new_team"] = t
        except Exception:
            pass

if drop_idx:
    out = out.drop(index=drop_idx)

# ── Collision resolution: same canonical ID twice in one frame → keep best ──
ppl2 = out["class_id"].isin(PEOPLE)
coll = (out[ppl2].groupby(["frame", "new_id"]).size()
        .reset_index(name="k").query("k > 1"))
if len(coll):
    out["_p"] = out["class_id"].isin(PEOPLE)
    keep = (out[out["_p"]]
            .sort_values("conf", ascending=False)
            .drop_duplicates(["frame", "new_id"], keep="first").index)
    out  = pd.concat([out[~out["_p"]], out.loc[keep]]).sort_index()
    out  = out.drop(columns="_p")

out["display_track_id"] = out["new_id"]
out["team_id"]          = out["new_team"]
out = out.drop(columns=["new_id", "new_team"])

# ── Gap interpolation ─────────────────────────────────────────────────────────
def _interp(d, max_gap=MAX_GAP):
    d = d.copy()
    if "track_filled" not in d.columns:
        d["track_filled"] = 0
    cols_lin = ["x1","y1","x2","y2","x_m","y_m","cx","cy"]
    cols_lin = [c for c in cols_lin if c in d.columns]
    new_rows = []
    ppl = d[d["class_id"].isin(PEOPLE)]
    for cid, g in ppl.groupby("display_track_id"):
        g = g.sort_values("frame")
        frames = g["frame"].values
        for a, b in zip(frames[:-1], frames[1:]):
            gap = int(b - a)
            if 1 < gap <= max_gap:
                ra = g[g["frame"] == a].iloc[0]
                rb = g[g["frame"] == b].iloc[0]
                for k in range(1, gap):
                    t   = k / gap
                    row = ra.copy()
                    row["frame"] = a + k
                    for c in cols_lin:
                        if pd.notna(ra[c]) and pd.notna(rb[c]):
                            row[c] = ra[c] + t * (rb[c] - ra[c])
                    row["conf"]         = 0.0
                    row["detector_ran"] = 0
                    row["track_filled"] = 1
                    new_rows.append(row)
    if new_rows:
        d = pd.concat([d, pd.DataFrame(new_rows)], ignore_index=True)
    return d.sort_values(["frame","display_track_id"]).reset_index(drop=True)

before = len(out)
out = _interp(out)
filled = int(out.get("track_filled", pd.Series([0])).sum())

# ── Save + download ───────────────────────────────────────────────────────────
out.to_csv(CLEAN_CSV, index=False)
print(f"\n✅ Saved → {CLEAN_CSV}")
ppl_out = out[out["class_id"].isin(PEOPLE)]
print(f"   Rows: {len(out):,}  |  Interpolated: {filled}")
print(f"   Canonical IDs: {ppl_out.display_track_id.nunique()}")
print(f"   Frame collisions resolved: {len(coll)}")
print()
print("Rows per canonical ID:")
print(ppl_out.groupby("display_track_id")["frame"].nunique()
      .sort_values(ascending=False).to_string())

colab_files.download(CLEAN_CSV)